### Chapter 5.3 - Forward Propagation, Backward Propagation, and Computational Graphs

This notebook explains how values flow forward through a network and how gradients flow backward through the recorded computation graph.


In [ ]:
import math
import random
import numpy as np
import torch


### 5.3.1 Forward Propagation

#### 1. Intuition

Forward propagation is the process of computing outputs from inputs by running the model operations in order.

Intermediate values are values created between input and output, such as hidden activations.

#### 2. Why this exists

The loss cannot be computed until predictions exist, and predictions come from the forward pass.


#### 3. Examples

A tiny forward pass through one hidden layer.


In [ ]:
X = torch.tensor([[1.0, 2.0]])
W1 = torch.randn(2, 3)
b1 = torch.zeros(3)
W2 = torch.randn(3, 1)
b2 = torch.zeros(1)
H = torch.relu(X @ W1 + b1)
y_hat = H @ W2 + b2
y_hat


#### 4. Step-by-step breakdown

The input `X` is used first.

`X @ W1 + b1` computes hidden pre-activations.

`torch.relu` creates hidden activations.

`H @ W2 + b2` computes the final output.

The forward pass stores values that backpropagation may need later.

#### 5. Connection to ML systems

Training loops begin each step with forward propagation, then use the result to compute loss.

#### 6. Common confusion points

- Forward propagation computes predictions only.
- It does not update parameters by itself.
- Intermediate values matter for gradient computation.
- Layer order determines computation order.


### 5.3.2 Computational Graph of Forward Propagation

#### 1. Intuition

A computational graph records how values are produced from other values.

Nodes are values or operations. Edges show dependency: which value was used to make another value.

#### 2. Why this exists

The graph lets automatic differentiation apply the chain rule systematically.


#### 3. Examples

A small graph-like dependency list.


In [ ]:
graph = [
    "X -> hidden linear",
    "hidden linear -> ReLU",
    "ReLU -> output linear",
    "output linear -> loss",
]
graph


PyTorch records tensor operations when gradients are required.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x * x
z = y + 3
z.grad_fn is not None


#### 4. Step-by-step breakdown

The dependency list is a human-readable graph.

In PyTorch, `requires_grad=True` asks autograd to track operations involving `x`.

`y` and `z` remember how they were computed.

`grad_fn` is PyTorch's stored backward-function reference for non-leaf tensors.

#### 5. Connection to ML systems

Deep learning frameworks build these graphs dynamically during ordinary tensor computation.

#### 6. Common confusion points

- A graph is about dependencies, not visual appearance.
- Leaf tensors are created directly by the user.
- Non-leaf tensors are produced by operations.
- The graph is used for gradients after the loss is computed.


### 5.3.3 Backpropagation

#### 1. Intuition

Backpropagation computes gradients by moving backward from the loss through the computational graph.

It applies the chain rule: multiply local sensitivities along dependency paths.

#### 2. Why this exists

Parameters can be improved only if we know how changing them would change the loss.


#### 3. Examples

PyTorch backpropagation through a small computation.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x * x
loss = y + 1
loss.backward()
x.grad


Manual chain rule for the same computation.


In [ ]:
x_value = 2.0
dloss_dy = 1.0
dy_dx = 2 * x_value
dloss_dx = dloss_dy * dy_dx
dloss_dx


#### 4. Step-by-step breakdown

`loss.backward()` starts from the scalar loss.

PyTorch traces back through `loss = y + 1` and `y = x * x`.

`x.grad` stores the derivative of loss with respect to `x`.

The manual calculation multiplies the local derivative from each step.

#### 5. Connection to ML systems

Backpropagation is the core algorithm that makes neural network training practical.

#### 6. Common confusion points

- Backpropagation computes gradients, not parameter updates.
- The optimizer uses gradients after backpropagation.
- Gradients accumulate unless cleared.
- The chain rule is the mathematical basis.


### 5.3.4 Training Neural Networks

#### 1. Intuition

Training a neural network repeats forward propagation, loss computation, backpropagation, and parameter updates.

A parameter update changes weights and biases using an optimizer.

#### 2. Why this exists

The full loop is where all pieces connect into learning.


#### 3. Examples

One tiny neural-network training step.


In [ ]:
net = torch.nn.Sequential(torch.nn.Linear(2, 3), torch.nn.ReLU(), torch.nn.Linear(3, 1))
X = torch.randn(4, 2)
y = torch.randn(4, 1)
loss_fn = torch.nn.MSELoss()
opt = torch.optim.SGD(net.parameters(), lr=0.1)
opt.zero_grad()
loss = loss_fn(net(X), y)
loss.backward()
opt.step()
loss


#### 4. Step-by-step breakdown

The model computes predictions with `net(X)`.

The loss compares predictions with labels.

`zero_grad()` clears old gradients.

`backward()` computes new gradients.

`step()` updates parameters.

#### 5. Connection to ML systems

This control flow is shared by MLPs, CNNs, RNNs, and transformers, even though their layers differ.

#### 6. Common confusion points

- Training is a repeated process, not one forward pass.
- Clearing gradients is necessary before the next backward pass.
- Loss must connect to parameters through the graph.
- Optimizer steps should happen after gradients exist.


### 5.3.5 Summary

#### 1. Intuition

Forward propagation computes predictions. Backpropagation computes gradients. The optimizer updates parameters.

The computational graph connects these steps.

#### 2. Why this exists

Understanding this flow removes much of the mystery from deep learning training loops.


#### 3. Examples

The repeated training sequence.


In [ ]:
sequence = [
    "forward",
    "loss",
    "zero gradients",
    "backward",
    "optimizer step",
]
sequence


#### 4. Step-by-step breakdown

The sequence names what happens in one training iteration.

In many PyTorch loops, `zero_grad` appears before forward or before backward.

The important rule is that old gradients should not pollute the new update.

#### 5. Connection to ML systems

Later training frameworks package this sequence, but the same execution order remains underneath.

#### 6. Common confusion points

- The graph records how tensors were computed.
- Backward uses the graph to compute gradients.
- The optimizer changes parameters.
- Debug training by checking each step separately.


### 5.3.6 Exercises

#### 1. Intuition

These exercises practice tracing values and gradients.

#### 2. Why this exists

Small graphs make it easier to understand large neural networks.


#### 3. Examples

Exercise 1: compute a gradient through two operations.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = 2 * x
loss = y * y
loss.backward()
x.grad


Exercise 2: list the training loop steps from memory.


In [ ]:
steps = ["forward", "loss", "backward", "update"]
steps


#### 4. Step-by-step breakdown

Exercise 1 checks chain-rule tracking.

Exercise 2 checks control-flow memory.

Both are core skills for reading training code.

#### 5. Connection to ML systems

These same ideas appear in all later deep learning chapters.

#### 6. Common confusion points

- Gradients are local to the current graph and values.
- A scalar loss makes `.backward()` easiest.
- Parameter updates are separate from gradient computation.
- Always know what is stored in memory during training.
